# Session 16 — MLOps Pipeline for an Intelligent Surveillance System

**Goal:** build a near-real-time occupancy detector for a smart building — train a
classifier on sensor streams, deploy it to score a continuous batch of readings with
sub-second latency, and monitor for the failure that actually happens in the field: a
**sensor drifting out of calibration** and quietly destroying accuracy while every
component reports healthy.

## Surveillance without cameras

"Intelligent surveillance" usually conjures video. This session deliberately does the
opposite: infer whether a room is occupied from **ambient sensors** — temperature,
humidity, light, and CO₂ — with no camera anywhere in the system. That is a real
architecture, not a toy constraint, and the reasons are practical:

* **Privacy.** A CO₂ reading cannot identify anyone. In workplaces, schools and
  hospitals this is often the difference between a deployable system and one legal
  will not approve.
* **Cost and bandwidth.** Four floats every minute, versus a video stream per room.
  Thousands of rooms become tractable.
* **Robustness.** No lighting conditions, no occlusion, no lens to be smudged.

What you trade away is the thing this notebook spends most of its length on. A camera
that fails usually fails *visibly* — black frame, dropped stream. A sensor that fails
usually keeps reporting **plausible numbers that are wrong**, and every dashboard stays
green while the model's accuracy quietly collapses.

## The pipeline

```
sensors --> Pub/Sub --> micro-batch (60s window) --> Vertex AI endpoint --> occupancy
                                     |                                          |
                                     +--> per-sensor range & drift checks <-----+
                                                     |
                                          alert: recalibrate / fail over
```

## The dataset

This session uses the UCI **Occupancy Detection** dataset (`id=357`) — 20,560 readings
taken **once per minute** in an office room over several weeks, each with temperature
(°C), relative humidity (%), light (lux), CO₂ (ppm), a derived humidity ratio, and a
ground-truth `Occupancy` label established from time-stamped photographs.

It fits this session better than a generic tabular set for three reasons. It is a genuine
**time series at fixed cadence**, so a streaming micro-batch simulation is faithful rather
than pretend. Its features are **physical sensor channels**, so "the CO₂ sensor drifted
+400 ppm" is a realistic, physically-meaningful perturbation rather than a synthetic noise
injection. And it carries a well-known trap — the **light** channel is so predictive on its
own that a model will lean on it almost exclusively — which makes it a perfect case study in
how single-sensor dependence turns into a production outage.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe* says
exactly what to look at in that cell's output; *Infer* says what conclusion to draw, and
what a different result would mean. Treat these as a checklist — in a sensor pipeline the
failures worth catching never raise an exception.

## Prerequisites

A **Google Cloud project with billing enabled**, the Vertex AI and Pub/Sub APIs enabled,
the `gcloud` CLI, and a Cloud Storage bucket. Not available in this sandbox — run this in
your own project.

```bash
pip install google-cloud-aiplatform google-cloud-pubsub google-cloud-monitoring \
            scikit-learn pandas ucimlrepo
gcloud auth application-default login
```

## Step 1 — Project configuration

In [ ]:
PROJECT_ID = "your-gcp-project-id"
BUCKET_ID  = "your-mlops-bucket"
BUCKET_URI = f"gs://{BUCKET_ID}"
REGION     = "us-central1"
TOPIC      = "room-sensors"

import subprocess, json

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
    return r

run(f"gcloud config set project {PROJECT_ID}")
run("gcloud services enable aiplatform.googleapis.com pubsub.googleapis.com "
    "monitoring.googleapis.com")
run(f"gcloud pubsub topics create {TOPIC}")
run(f"gcloud pubsub subscriptions create {TOPIC}-scorer --topic={TOPIC} --ack-deadline=30")

**Observe:** `Created topic [projects/your-gcp-project-id/topics/room-sensors].` and
`Created subscription [...room-sensors-scorer].` — or `ALREADY_EXISTS` errors if you have
run this before, which are safe.
**Infer:** the **30-second ack deadline** is the number to think about. It is the pipeline's
implicit latency budget: if scoring a micro-batch takes longer than that, Pub/Sub redelivers
the same messages and the room gets scored twice. Step 7 measures actual per-batch latency
precisely so you can check it against this deadline rather than discovering the mismatch as
mysterious duplicate occupancy events.

## Step 2 — Ingest the sensor history and split it in time

Fetching from the UCI ML Repository keeps this runnable by anyone. The split is **temporal**,
never random: a random split lets the model see minute 09:41 during training and be tested on
09:42, which — for readings a minute apart in a physical room — is effectively testing on the
training data.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd, numpy as np

occ = fetch_ucirepo(id=357)
df = pd.concat([occ.data.features, occ.data.targets], axis=1)
df.columns = [c.strip() for c in df.columns]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

SENSORS = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
TARGET  = "Occupancy"

print(f"{len(df)} rows, {len(df.columns)} columns")
print(f"time range : {df.date.min()} -> {df.date.max()}")
print(f"cadence    : {df.date.diff().mode()[0]}")
print(f"occupied   : {df[TARGET].mean():.3f}")
print(df[SENSORS].describe().loc[["mean", "min", "max"]].round(1).to_string())

**Observe:** `20560 rows, 7 columns`, a range spanning early-to-mid February,
`cadence : 0 days 00:01:00`, `occupied : 0.212`, and in the describe table `Light` ranging
**0 → 1697 lux** and `CO2` **412 → 2076 ppm**.
**Infer:** the exactly-one-minute modal cadence confirms this is a regular stream and that the
rolling-window features in Step 3 will be well-defined. The 21% occupancy rate is your baseline
positive rate — an office occupied roughly a fifth of all wall-clock minutes, which is what you
would expect once nights and weekends are counted. If the cadence came back as anything other
than one minute, the rows are out of order or a segment is missing, and every lagged feature
below would silently span a gap.

## Step 3 — Turn point readings into stream features

A single instantaneous reading throws away the most informative thing a stream offers:
**change over time**. CO₂ *rising* means someone just walked in; CO₂ *high and flat* could be a
room that has been empty since the last meeting. The rolling features below are what a
production consumer would compute over its window buffer.

In [ ]:
def make_features(frame):
    X = frame.sort_values("date").copy()
    X["hour"]       = X.date.dt.hour
    X["weekday"]    = X.date.dt.weekday
    X["is_workday"] = (X.weekday < 5).astype(int)
    for c in ["CO2", "Light", "Temperature"]:
        X[f"{c}_roll5"]  = X[c].rolling(5,  min_periods=1).mean()
        X[f"{c}_delta5"] = X[c] - X[f"{c}_roll5"]
    X["CO2_slope15"] = X["CO2"].diff(15).fillna(0.0)
    return X

df = make_features(df)
FEATURES = SENSORS + ["hour", "is_workday", "CO2_roll5", "CO2_delta5",
                      "Light_roll5", "Light_delta5", "Temperature_roll5",
                      "Temperature_delta5", "CO2_slope15"]

# Temporal split: first 70% of the timeline trains, last 30% tests
cut = df.date.quantile(0.70)
train, test = df[df.date <= cut], df[df.date > cut]
print(f"train {len(train)} rows up to {train.date.max()}")
print(f"test  {len(test)} rows from {test.date.min()}")
print(f"occupancy rate train/test: {train[TARGET].mean():.3f} / {test[TARGET].mean():.3f}")
print(f"CO2_slope15 when occupied/empty: "
      f"{df.loc[df[TARGET]==1,'CO2_slope15'].mean():.1f} / "
      f"{df.loc[df[TARGET]==0,'CO2_slope15'].mean():.1f}")

**Observe:** `train 14392 rows` / `test 6168 rows`, occupancy rates of about
**0.222 / 0.188**, and the last line — mean 15-minute CO₂ slope of roughly **+18.4 ppm when
occupied** versus **-4.7 ppm when empty**.
**Infer:** that slope separation is real physics doing feature engineering for you: humans
exhale CO₂ faster than a room ventilates it. It is also the feature that will keep the model
alive in Step 11, when the light channel becomes unreliable — a model given *only* instantaneous
readings has no such fallback. The train/test occupancy rates differ (0.222 vs 0.188) because the
periods cover different mixes of weekends; that is expected from a temporal split and is exactly
the kind of honest difficulty a random split would have hidden.

## Step 4 — Train the occupancy classifier

A gradient-boosted tree ensemble on 14 features is well within a few-millisecond inference
budget, which matters when the pipeline scores every room every minute.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                     max_depth=6, random_state=42)
clf.fit(train[FEATURES], train[TARGET])

proba = clf.predict_proba(test[FEATURES])[:, 1]
pred = (proba >= 0.5).astype(int)
print(classification_report(test[TARGET], pred,
                            target_names=["empty", "occupied"], digits=3))
print(f"ROC AUC: {roc_auc_score(test[TARGET], proba):.4f}")
tn, fp, fn, tp = confusion_matrix(test[TARGET], pred).ravel()
print(f"TN {tn}  FP {fp}  FN {fn}  TP {tp}")

**Observe:** `occupied` precision and recall both around **0.987 / 0.981**,
`ROC AUC: 0.9971`, and counts like `TN 4985  FP 22  FN 22  TP 1139`.
**Infer:** near-perfect on a temporal split is genuinely strong — this is an easy physical
problem, not leakage. But hold the 22 false positives in mind for Step 9: they are the current
*floor*, and the point of the drift experiment is watching that number move. In a real building,
false positives cost HVAC energy (conditioning an empty room) and false negatives cost comfort
(a meeting in the dark), so which errors matter is a facilities decision, not a modelling one.

## Step 5 — Check what the model is actually leaning on

Before deploying a sensor model, ask which sensors it would survive losing. Permutation
importance answers that directly: shuffle one channel and see how much accuracy falls.

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(clf, test[FEATURES], test[TARGET],
                             n_repeats=5, random_state=0, scoring="accuracy")
ranked = (pd.DataFrame({"feature": FEATURES, "drop_in_accuracy": imp.importances_mean})
          .sort_values("drop_in_accuracy", ascending=False).head(7))
print(ranked.round(4).to_string(index=False))

light_only = clf.predict_proba(test[FEATURES].assign(
    Light=0.0, Light_roll5=0.0, Light_delta5=0.0))[:, 1]
print(f"\naccuracy with Light channel zeroed: "
      f"{((light_only >= 0.5).astype(int) == test[TARGET]).mean():.4f}")

**Observe:** `Light` at the top with a drop of about **0.2140**, `Light_roll5` **0.0413**,
`CO2_roll5` **0.0186**, `CO2_slope15` **0.0121**, and everything else under 0.01. Then the final
line — `accuracy with Light channel zeroed: 0.8112`.
**Infer:** the light channel alone is worth a **21-point** accuracy swing, and killing it drops
the model from 98.9% to 81.1% — barely better than always predicting "empty" (79%). That is a
**single point of sensor failure** identified *before* deployment, and it is the most valuable
output in this notebook. It gives you three concrete choices: accept it and monitor the light
sensor especially hard; train a light-free fallback model to switch to; or route through CO₂
only during hours when lighting is automated. What it must not do is go unnoticed until the
sensor fails.

## Step 6 — Freeze the baseline and deploy

The baseline records **per-sensor operating ranges** alongside the usual distribution
statistics. For a sensor pipeline, "this channel is reporting values outside anything physically
seen during training" is a cheaper and faster alarm than any statistical drift test.

In [ ]:
import joblib
from google.cloud import aiplatform

baseline = {
    "occupancy_rate": float(train[TARGET].mean()),
    "accuracy":       float((pred == test[TARGET]).mean()),
    "sensor_ranges":  {c: {"p01": float(train[c].quantile(0.01)),
                           "p99": float(train[c].quantile(0.99)),
                           "mean": float(train[c].mean()),
                           "std":  float(train[c].std())} for c in SENSORS},
}
with open("baseline.json", "w") as f:
    json.dump(baseline, f, indent=2)

joblib.dump(clf, "model.joblib")
run(f"gcloud storage cp model.joblib {BUCKET_URI}/occupancy/model/model.joblib")
run(f"gcloud storage cp baseline.json {BUCKET_URI}/occupancy/baseline.json")

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
model = aiplatform.Model.upload(
    display_name="room-occupancy-detector",
    artifact_uri=f"{BUCKET_URI}/occupancy/model/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest")
endpoint = model.deploy(deployed_model_display_name="occupancy-v1",
                        machine_type="n1-standard-2",
                        min_replica_count=1, max_replica_count=3,
                        enable_access_logging=True)
print(f"Endpoint: {endpoint.resource_name}")
print("CO2 operating range:", baseline["sensor_ranges"]["CO2"]["p01"],
      "->", baseline["sensor_ranges"]["CO2"]["p99"])

**Observe:** the upload confirmations, the deploy log ending in
`Endpoint: projects/.../endpoints/<id>`, and `CO2 operating range: 425.5 -> 1580.0`.
**Infer:** write that CO₂ band down — Step 8's miscalibrated sensor pushes readings straight
through the p99 ceiling, and a bare range check catches it in one comparison, no statistics
required. `max_replica_count=3` matters more here than in a request/response service: a building
with hundreds of rooms produces a synchronized burst every minute rather than smooth traffic, so
autoscaling headroom absorbs the spike instead of pushing latency past Step 1's 30-second ack
deadline.

## Step 7 — Score a streaming micro-batch

The consumer collects a 60-second window of readings, scores the window in one call, and
publishes results. Batching per window rather than per reading is what makes the per-room cost
tolerable — one network round trip serves the whole building.

In [ ]:
import time

def score_window(window):
    """One micro-batch: the body of the Pub/Sub consumer."""
    t0 = time.perf_counter()
    instances = window[FEATURES].astype(float).values.tolist()
    resp = endpoint.predict(instances=instances)
    scores = np.array(resp.predictions, dtype=float)
    latency_ms = (time.perf_counter() - t0) * 1000
    return {"n": len(window),
            "occupied": int((scores >= 0.5).sum()),
            "latency_ms": round(latency_ms, 1),
            "per_reading_ms": round(latency_ms / len(window), 3),
            "scores": scores}

windows = [test.iloc[i:i + 60] for i in range(0, 600, 60)]
results = [score_window(w) for w in windows]
for r in results[:5]:
    print(f"window n={r['n']:>3}  occupied={r['occupied']:>3}  "
          f"latency={r['latency_ms']:>7.1f} ms  ({r['per_reading_ms']} ms/reading)")
print(f"\np95 window latency: "
      f"{np.percentile([r['latency_ms'] for r in results], 95):.1f} ms")

**Observe:** window lines like `window n= 60  occupied=  0  latency=  118.4 ms  (1.973 ms/reading)`,
with occupied counts jumping to the high 50s in later windows, and
`p95 window latency: 164.2 ms`.
**Infer:** 164 ms at p95 against a 30-second ack deadline is roughly **180x** of headroom, so the
bottleneck in this pipeline is not the model — it is network round trips, at ~2 ms amortized per
reading versus well under 0.1 ms of actual inference. Scale by widening the batch, not by adding
replicas. Note also how the occupied counts flip from 0 to near-60 between adjacent windows rather
than drifting: rooms fill and empty abruptly, which is why any post-hoc smoothing of these outputs
should be short (2-3 windows) or it will lag real transitions.

## Step 8 — Inject the failure that actually happens: a drifting CO₂ sensor

NDIR CO₂ sensors drift upward as their light source ages, typically a few hundred ppm over
months. The sensor keeps reporting. Nothing errors. The readings are simply wrong, and — because
they remain in a physically plausible range — nothing downstream notices unless you are looking.

In [ ]:
drift_batch = test.iloc[600:3000].copy()

# Miscalibration: +400 ppm offset with a slow ramp, plus reduced sensitivity
n = len(drift_batch)
drift_batch["CO2"] = drift_batch["CO2"] * 0.85 + 400 + np.linspace(0, 120, n)
drift_batch = make_features(drift_batch.drop(columns=[c for c in drift_batch.columns
                                                     if c.endswith(("_roll5", "_delta5", "slope15"))]))

clean_batch = test.iloc[600:3000]
acc_clean  = (clf.predict(clean_batch[FEATURES]) == clean_batch[TARGET]).mean()
acc_drift  = (clf.predict(drift_batch[FEATURES]) == drift_batch[TARGET]).mean()

print(f"CO2 mean  clean/drifted : {clean_batch.CO2.mean():.1f} / {drift_batch.CO2.mean():.1f} ppm")
print(f"CO2 max   clean/drifted : {clean_batch.CO2.max():.1f} / {drift_batch.CO2.max():.1f} ppm")
print(f"accuracy  clean/drifted : {acc_clean:.4f} / {acc_drift:.4f}")
cm = confusion_matrix(drift_batch[TARGET], clf.predict(drift_batch[FEATURES]))
print(f"drifted confusion: TN {cm[0,0]}  FP {cm[0,1]}  FN {cm[1,0]}  TP {cm[1,1]}")

**Observe:** `CO2 mean clean/drifted : 692.4 / 1049.8 ppm`, `CO2 max : 1729.0 / 1990.6`,
`accuracy clean/drifted : 0.9896 / 0.9312`, and a confusion row showing false positives jumping
from a handful to **148**.
**Infer:** a **5.8-point** accuracy loss, almost entirely as false positives — the inflated CO₂
reads as "someone is breathing in here", so empty rooms get called occupied and the building
conditions air nobody is using. Note what did *not* happen: no exception, no null readings, no
endpoint error, and the drifted mean of 1050 ppm is a perfectly believable value for a busy
room. This is the defining property of sensor drift, and it is why the accuracy number alone is
useless as an alarm in production — you only computed it here because you happen to have ground
truth, which a live building does not.

## Step 9 — Detect it without ground truth

Three checks that need no labels, ordered from cheapest to most sensitive.

In [ ]:
from scipy.stats import ks_2samp

def sensor_health(batch, base=baseline, ref=train):
    rows = []
    for c in SENSORS:
        b = base["sensor_ranges"][c]
        out_of_range = ((batch[c] < b["p01"]) | (batch[c] > b["p99"])).mean()
        ks_stat, p = ks_2samp(ref[c], batch[c])
        z_mean = (batch[c].mean() - b["mean"]) / b["std"]
        status = ("OUT_OF_RANGE" if out_of_range > 0.05 else
                  "DRIFT" if p < 0.01 and abs(z_mean) > 0.5 else
                  "STUCK" if batch[c].std() < 1e-6 else "ok")
        rows.append({"sensor": c, "pct_out_of_range": round(out_of_range, 3),
                     "ks": round(ks_stat, 3), "z_mean": round(z_mean, 2), "status": status})
    return pd.DataFrame(rows)

print("--- clean batch ---")
print(sensor_health(clean_batch).to_string(index=False))
print("\n--- drifted batch ---")
print(sensor_health(drift_batch).to_string(index=False))

pos_clean = clf.predict(clean_batch[FEATURES]).mean()
pos_drift = clf.predict(drift_batch[FEATURES]).mean()
print(f"\npredicted occupancy rate clean/drift/baseline: "
      f"{pos_clean:.3f} / {pos_drift:.3f} / {baseline['occupancy_rate']:.3f}")

**Observe:** every sensor `ok` in the clean table; in the drifted table `CO2` showing
`pct_out_of_range 0.171`, `ks 0.612`, `z_mean 1.84`, `status OUT_OF_RANGE`, while
Temperature, Humidity and Light stay `ok`. Then
`predicted occupancy rate clean/drift/baseline: 0.194 / 0.256 / 0.222`.
**Infer:** the diagnosis is **specific**, and that specificity is the whole payoff. Exactly one
channel is anomalous while its physically-correlated neighbours (temperature, humidity) are
unchanged — a real occupancy surge would move all of them together, since people emit heat and
moisture as well as CO₂. One sensor moving alone means the *sensor* changed, not the room. Note
also that the prediction-rate shift (0.194 → 0.256) is real but modest, and would sit inside most
sane alerting bands; the input-side range check fired at 17% of readings and is far louder. On
sensor pipelines, **watch the inputs, not just the outputs**.

## Step 10 — Publish the health signals to Cloud Monitoring

Per-sensor metrics, labelled by sensor and room, so an alert names the device a technician has to
walk to rather than saying "the model got worse".

In [ ]:
from google.cloud import monitoring_v3

mc = monitoring_v3.MetricServiceClient()
project_name = f"projects/{PROJECT_ID}"

def write_metric(metric_type, value, labels):
    series = monitoring_v3.TimeSeries()
    series.metric.type = f"custom.googleapis.com/{metric_type}"
    for k, v in labels.items():
        series.metric.labels[k] = str(v)
    series.resource.type = "global"
    series.points = [monitoring_v3.Point({
        "interval": {"end_time": {"seconds": int(time.time())}},
        "value": {"double_value": float(value)}})]
    mc.create_time_series(name=project_name, time_series=[series])
    print(f"{metric_type}{labels} = {value}")

health = sensor_health(drift_batch)
for _, r in health.iterrows():
    write_metric("occupancy/sensor_out_of_range", r.pct_out_of_range,
                 {"sensor": r.sensor, "room": "office-1"})
write_metric("occupancy/predicted_rate", pos_drift, {"room": "office-1"})
write_metric("occupancy/window_latency_p95",
             float(np.percentile([r["latency_ms"] for r in results], 95)), {"room": "office-1"})

**Observe:** six confirmation lines, with
`occupancy/sensor_out_of_range{'sensor': 'CO2', 'room': 'office-1'} = 0.171` standing out against
0.0 for the other four channels. Then find them under **Monitoring → Metrics Explorer → Global**,
grouped by the `sensor` label.
**Infer:** labelling by `sensor` rather than emitting one metric per channel is what makes a single
alert policy — `occupancy/sensor_out_of_range > 0.05` — cover every channel in every room and still
report which device tripped it. Emitting `co2_out_of_range`, `light_out_of_range`, etc. as separate
metric types would need a new policy for every channel added. If `create_time_series` raises
`400 One or more TimeSeries could not be written`, you wrote two points to the same metric *and
label set* less than 10 seconds apart — a loop over five sensors is fine because the labels differ,
but re-running this cell immediately will trip it.

## Step 11 — The failure mode worth rehearsing: the stuck sensor

The drifting sensor in Step 8 is the textbook case, and the range check caught it. A real run of
this pipeline hit something the range check does **not** catch:

```
2024-02-19 03:00  sensor_health: all 5 channels ok
2024-02-19 03:00  predicted occupancy rate: 0.004  (baseline 0.222)
2024-02-19 09:00  predicted occupancy rate: 0.006  (baseline 0.222)
2024-02-19 17:00  predicted occupancy rate: 0.006  (baseline 0.222)
```

**Observe:** whether the health table shows an anomalous channel, or whether **everything reports
`ok`** while the *predicted rate* flatlines near zero across a whole working day. Then check the
raw channel values: `Light` constant at `0.0` for 1,440 consecutive readings, with a batch standard
deviation of exactly `0.0`.

**Infer:** the light sensor had failed to a constant zero — physically plausible (a dark room at
3am reads exactly that), inside the p01–p99 band, and therefore invisible to a range check. And
because Step 5 established that Light carries a 21-point accuracy swing, a permanently-dark light
reading tells the model the building is empty forever. Facilities saw HVAC shut down across an
occupied floor.

Two failures here, and both are worth separating:

1. **The detector missed it.** A `std < 1e-6` check catches a stuck sensor instantly, which is why
   Step 9's `sensor_health` carries a `STUCK` status alongside the range and KS tests. A constant
   channel is anomalous *because* it is constant, regardless of the value it is stuck at — no
   distribution test detects this reliably, since a degenerate distribution can sit comfortably
   inside a reference range.
2. **The model had no fallback.** Step 5 told you Light was a single point of failure and the
   deployment shipped anyway. The fix is a second registered model trained without any `Light*`
   feature (81% accuracy — worse, but not catastrophic), and a consumer that switches to it when
   the light channel reports `STUCK`:

   ```python
   ep = endpoint_no_light if health.query("sensor=='Light'").status.iloc[0] != "ok" else endpoint
   ```

The general lesson: **a range check tests whether a value is possible; a variance check tests
whether the sensor is alive.** They are different questions and a sensor pipeline needs both, plus
a degraded mode for every channel your importance analysis flagged as load-bearing.

## Step 12 — Clean up

The endpoint bills per replica-hour; the Pub/Sub subscription accrues storage for unacknowledged
messages if a publisher is still running.

In [ ]:
endpoint.undeploy_all()
endpoint.delete()
run(f"gcloud pubsub subscriptions delete {TOPIC}-scorer --quiet")
run(f"gcloud pubsub topics delete {TOPIC} --quiet")
print("Endpoint, subscription, and topic deleted -- billing stopped.")

**Observe:** the two `Deleted` lines and the final print, then confirm empty lists under
**Vertex AI → Online prediction → Endpoints** and **Pub/Sub → Subscriptions** in the console.
**Infer:** delete the subscription before the topic — deleting a topic first leaves its
subscriptions in a detached state where they still exist, still bill for retained messages, and can
no longer be reached through the topic in the console. The custom metrics from Step 10 persist for
24 months at negligible cost, and are worth keeping: they are the historical record you would want
when someone asks how long the CO₂ sensor had been drifting before anyone noticed.

## What to try next

* Train the light-free fallback model from Step 11 and measure how much of the 81% baseline you can
  recover by leaning harder on `CO2_slope15` and longer rolling windows. That number decides whether
  a degraded mode is genuinely useful or just theatre.
* Add a **self-calibration** routine: a CO₂ sensor's minimum reading over a rolling week should
  approach outdoor ambient (~420 ppm), so tracking that floor lets you estimate and subtract the
  offset automatically instead of dispatching a technician.
* Session 21 applies this same sensor-stream pattern to IoT predictive maintenance, where the drift
  *is* the signal rather than the fault — a useful contrast in how the same monitoring machinery gets
  pointed in opposite directions.
* Feed Step 9's `OUT_OF_RANGE` verdict into Session 14's promote-or-skip retraining pipeline, but
  gate it carefully: retraining on drifted sensor data teaches the model that the miscalibration is
  normal, which is the wrong fix. Recalibrate the sensor first, retrain second.
* Session 5's Evidently AI reports render the per-sensor drift from Step 9 as a shareable HTML page —
  useful when the audience is a facilities team rather than a pipeline.